# SQL Error Review

Every failed **Submit** in `nb01_sql_practice.ipynb` (a SQL error OR a wrong answer on the hidden test) is logged to `data/outputs/errors/` with the full problem, your SQL, the error or your-vs-expected output, and the correct `answer_key`.

This notebook ranks **which type + subtype patterns you fail most** (frequency analysis), points each one to its **tab / recipe / leaf in `sql_problem_patterns.html`** so you can study the pattern, and shows every mistake next to the right answer. The optional **Analyze with Claude** button reads your failures and names the recurring *kinds* of mistakes (NULL handling, grain, joins, filters, ordering) with a prioritized practice plan.

**How to use:** run setup → run the Overview cell → pick a Type / Subtype / Kind filter (or *All*) → click **Show overview**.


In [ ]:
import os, sys, html as _htmlmod
from urllib.parse import quote as _urlquote
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import sql_practice_utils as spu
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
ERRORS_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'errors')
os.makedirs(ERRORS_DIR, exist_ok=True)

def _find_playbook(start):
    d = start
    for _ in range(8):
        for cand in [os.path.join(d, 'sql', 'sql_problem_patterns.html'),
                     os.path.join(d, 'folders', 'sql', 'sql_problem_patterns.html')]:
            if os.path.exists(cand):
                return os.path.abspath(cand)
        d = os.path.dirname(d)
    return None
PLAYBOOK = _find_playbook(PROJECT_ROOT)

print('Errors dir:', ERRORS_DIR)
print('Logged failures so far:', len(spu.load_errors(ERRORS_DIR)))
print('Playbook:', PLAYBOOK or 'NOT FOUND (links will show the breadcrumb only)')


## Overview & recommendations

In [ ]:
def _esc(s):
    return _htmlmod.escape('' if s is None else str(s))

def _pb_link(pointer):
    """Breadcrumb (Tab > Recipe > Leaf) + a clickable deep link into the playbook."""
    if not pointer:
        return ''
    label = _esc(pointer.get('label', ''))
    anchor = pointer.get('anchor')
    if PLAYBOOK and anchor:
        href = 'file://' + _urlquote(PLAYBOOK) + '#' + anchor
        return f'\U0001F4D6 <a href="{href}" target="_blank">{label}</a>'
    return f'\U0001F4D6 {label}'

def _mini_table(cols, rows, max_rows=12):
    if not cols:
        return '<div style="color:#57606a;">(no result)</div>'
    th = ''.join(f'<th style="padding:3px 10px; border-bottom:2px solid #cbd5e1; text-align:left;">{_esc(c)}</th>' for c in cols)
    body = ''
    for r in (rows or [])[:max_rows]:
        tds = ''.join(f'<td style="padding:3px 10px;">{_esc(v)}</td>' for v in r)
        body += f'<tr>{tds}</tr>'
    more = '' if not rows or len(rows) <= max_rows else f'<div style="color:#57606a;">… {len(rows)-max_rows} more rows</div>'
    return f'<table style="border-collapse:collapse; font-size:0.9rem;"><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>{more}'

def _render_error(r):
    head = f"{r.get('title','(untitled)')} — {r.get('question_type')} / {r.get('subtype') or '(none)'} · {r.get('failure_kind')} · {str(r.get('logged_at',''))[:19]}"
    sub = None if r.get('subtype') in (None, '(none)') else r.get('subtype')
    ptr = spu.playbook_pointer(r.get('question_type'), sub)
    h = [f'<div style="border:1px solid #d0d7de; border-radius:6px; padding:10px; margin:8px 0;">']
    h.append(f'<div style="font-weight:600; margin-bottom:4px;">{_esc(head)}</div>')
    if ptr:
        h.append(f'<div style="margin-bottom:6px;">Study: {_pb_link(ptr)}</div>')
    if r.get('prompt'):
        h.append(f'<div style="color:#57606a; margin-bottom:6px;">{_esc(r["prompt"])}</div>')
    h.append('<div style="font-weight:600;">Your SQL</div>')
    h.append(f'<pre style="background:#f6f8fa; padding:8px; white-space:pre-wrap; border-radius:4px;">{_esc(r.get("user_solution",""))}</pre>')
    if r.get('failure_kind') == 'sql_error':
        h.append('<div style="font-weight:600; color:#cf222e;">Error</div>')
        h.append(f'<pre style="background:#ffebe9; padding:8px; white-space:pre-wrap; border-radius:4px;">{_esc(r.get("error_message",""))}</pre>')
    else:
        h.append('<div style="display:flex; gap:24px; flex-wrap:wrap;">')
        h.append('<div><div style="font-weight:600; color:#cf222e;">Your output</div>' + _mini_table(r.get('your_columns'), r.get('your_rows')) + '</div>')
        h.append('<div><div style="font-weight:600; color:#1a7f37;">Expected</div>' + _mini_table(r.get('test_expected_columns'), r.get('test_expected_rows')) + '</div>')
        h.append('</div>')
    h.append('<div style="font-weight:600; color:#1a7f37; margin-top:6px;">Correct answer (answer_key)</div>')
    h.append(f'<pre style="background:#dcfce7; padding:8px; white-space:pre-wrap; border-radius:4px;">{_esc(r.get("answer_key",""))}</pre>')
    h.append('</div>')
    display(HTML(''.join(h)))

def _type_options():
    recs = spu.load_errors(ERRORS_DIR)
    ts = sorted({r.get('question_type') for r in recs if r.get('question_type')})
    return [('All types', None)] + [(t, t) for t in ts]

type_dd = widgets.Dropdown(options=_type_options(), description='Type:')
subtype_dd = widgets.Dropdown(options=[('All subtypes', None)], description='Subtype:')
kind_dd = widgets.Dropdown(options=[('All kinds', None), ('Wrong answer', 'wrong_answer'), ('SQL error', 'sql_error')], description='Kind:')

def _refresh_subtypes(*_):
    recs = spu.load_errors(ERRORS_DIR, qtype=type_dd.value)
    subs = sorted({(r.get('subtype') or '(none)') for r in recs})
    subtype_dd.options = [('All subtypes', None)] + [(s, None if s == '(none)' else s) for s in subs]
type_dd.observe(_refresh_subtypes, names='value')

out = widgets.Output()

def show_overview(b=None):
    with out:
        clear_output(wait=True)
        recs = spu.load_errors(ERRORS_DIR, qtype=type_dd.value, subtype=subtype_dd.value, failure_kind=kind_dd.value)
        if not recs:
            print('No logged failures match this filter yet. Fail a Submit in nb01 to populate this.')
            return
        s = spu.summarize_errors(recs)
        print(f'Total failures (this filter): {s["total"]}')
        display(HTML('<b>Most-failed patterns (type + subtype) — drill these first</b>'))
        display(s['by_pattern'])
        rec = spu.recommend_practice(recs, top=5)
        if rec:
            items = ''
            for x in rec:
                items += (f'<li>{_esc(x["question_type"])} / {_esc(x["subtype"])} — {x["failures"]} failures'
                          f' &nbsp; {_pb_link(x.get("playbook"))}</li>')
            display(HTML(f'<b>Recommended practice order (with where to study):</b><ol>{items}</ol>'))
        display(HTML('<b>By question type</b>')); display(s['by_type'])
        display(HTML('<b>By failure kind</b>')); display(s['by_kind'])
        display(HTML('<hr><b>Details (newest first, up to 25)</b>'))
        for r in recs[:25]:
            _render_error(r)

btn = widgets.Button(description='Show overview', button_style='primary')
btn.on_click(show_overview)
display(widgets.VBox([widgets.HBox([type_dd, subtype_dd, kind_dd]), btn, out]))


## Optional: Claude blind-spot summary

Uses the current filter above. Sends your failed attempts (your SQL + the error or correct answer, plus each one's playbook location) to Claude and asks it to name the recurring **kinds** of mistakes and give a short practice plan that points back to the playbook recipes. Requires the `anthropic` package and an API key (same setup as nb01).

*Note on method:* with a personal error log, plain frequency counts (above) plus Claude's qualitative read are the right tools — a trained ML model would need far more data and would not beat counting which patterns you miss most.

In [ ]:
cout = widgets.Output()

def claude_blindspots(b=None):
    with cout:
        clear_output(wait=True)
        recs = spu.load_errors(ERRORS_DIR, qtype=type_dd.value, subtype=subtype_dd.value, failure_kind=kind_dd.value)
        if not recs:
            print('No failures to analyze for this filter.')
            return
        if not spu.init_claude():
            print('Claude client unavailable — check the anthropic package / API key.')
            return
        import json as _json
        items = []
        for r in recs[:30]:
            sub = None if r.get('subtype') in (None, '(none)') else r.get('subtype')
            ptr = spu.playbook_pointer(r.get('question_type'), sub)
            items.append({
                'question_type': r.get('question_type'),
                'subtype': r.get('subtype'),
                'failure_kind': r.get('failure_kind'),
                'title': r.get('title'),
                'playbook_location': (ptr or {}).get('label'),
                'user_solution': r.get('user_solution'),
                'error_message': r.get('error_message'),
                'answer_key': r.get('answer_key'),
            })
        sys_p = ('You are a SQL coach. Given a learner\'s FAILED attempts (their SQL, plus either the error '
                 'message or the correct answer_key, plus the playbook_location each one belongs to), do three things, '
                 'briefly and concretely: (1) name the recurring KINDS of mistakes (e.g., NULL handling, wrong grain / '
                 'GROUP BY, wrong join type, filter logic, ordering / limit); (2) say which type+subtype patterns they '
                 'struggle with most; (3) give a prioritized practice plan that names the playbook_location(s) to study.')
        user_p = 'Failures as JSON:\n' + _json.dumps(items, default=str)[:14000]
        txt = spu._call_claude(sys_p, user_p, max_tokens=1200)
        if txt:
            display(HTML(f'<div style="white-space:pre-wrap; line-height:1.5;">{_esc(txt)}</div>'))
        else:
            print('No response from Claude.')

cbtn = widgets.Button(description='Analyze with Claude', button_style='info')
cbtn.on_click(claude_blindspots)
display(widgets.VBox([cbtn, cout]))
